# Parte 1 - Taller Practico y Conceptual de PDI
## Modulo A: Analisis y Ecualizacion de Histogramas (7.5%)

Universidad de Antioquia - Procesamiento Digital de Imagenes - 2026-II

Imagen de trabajo: `im1.png` (incluida junto al notebook).


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

def mostrar(img, titulo, cmap=None):
    plt.figure(figsize=(7, 5))
    if img.ndim == 3:
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    else:
        plt.imshow(img, cmap=cmap or 'gray', vmin=0, vmax=255)
    plt.title(titulo)
    plt.axis('off')
    plt.show()

def histograma(img, canal=0, mascara=None):
    return cv2.calcHist([img], [canal], mascara, [256], [0, 256]).flatten()

img = cv2.imread('im1.png')
assert img is not None, 'No se encontro im1.png'
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
print('Imagen:', img.shape, '| OpenCV', cv2.__version__)

---
## Ejercicio A.1 - Calculo de Histogramas

El histograma $h(r_k) = n_k$ cuenta cuantos pixeles tienen intensidad $r_k$. Normalizado por el total $N$ se obtiene la PDF $p(r_k) = n_k / N$. Se calcula para escala de grises y para cada canal en RGB y HSV.


In [ ]:
mostrar(img, 'Imagen original')

plt.figure(figsize=(8, 3))
plt.plot(histograma(gray), color='k')
plt.title('Histograma en escala de grises')
plt.xlabel('Intensidad'); plt.ylabel('N. pixeles'); plt.xlim(0, 255)
plt.grid(alpha=0.3); plt.show()

print(f'Media: {gray.mean():.1f} | Desv. estandar: {gray.std():.1f}')
print(f'Pixeles oscuros (<64): {100*np.mean(gray < 64):.1f}%')

# Lectura: la masa esta en intensidades bajas -> imagen oscura.

In [ ]:
plt.figure(figsize=(8, 3))
for i, c, n in zip(range(3), ('b', 'g', 'r'), ('B', 'G', 'R')):
    plt.plot(histograma(img, i), color=c, label=n)
plt.title('Histogramas por canal - RGB'); plt.xlabel('Intensidad'); plt.ylabel('N. pixeles')
plt.xlim(0, 255); plt.legend(); plt.grid(alpha=0.3); plt.show()

plt.figure(figsize=(8, 3))
for i, c, n in zip(range(3), ('m', 'c', 'y'), ('H (Tono)', 'S (Sat.)', 'V (Brillo)')):
    plt.plot(histograma(hsv, i), color=c, label=n)
plt.title('Histogramas por canal - HSV'); plt.xlabel('Valor'); plt.ylabel('N. pixeles')
plt.xlim(0, 255); plt.legend(); plt.grid(alpha=0.3); plt.show()

# Lectura: R, G, B con medias casi iguales -> escena acromatica.
# En HSV, S baja -> poca saturacion; V repite el patron de grises -> escena oscura.

---
## Ejercicio A.2 - Ecualizacion Global vs CLAHE

**Global** (`cv2.equalizeHist`): una sola CDF $s = (L-1) \cdot \mathrm{CDF}(r)$ para toda la imagen. Maximiza el contraste global pero la pendiente $\propto p(r)$ sobre-amplifica el ruido.

**CLAHE** (`cv2.createCLAHE`): divide la imagen en celdas (`tileGridSize`) y ecualiza cada una con su propia CDF local, recortando el histograma segun `clipLimit` para acotar la ganancia. Se prueba con `clipLimit` 2.0 y 4.0, y `tileGridSize` (8,8) y (16,16).


In [ ]:
eq_global = cv2.equalizeHist(gray)

configs = [(2.0, (8, 8)), (2.0, (16, 16)), (4.0, (8, 8)), (4.0, (16, 16))]
clahe_results = {}
for clip, tile in configs:
    c = cv2.createCLAHE(clipLimit=clip, tileGridSize=tile).apply(gray)
    clahe_results[f'clip={clip}, tile={tile}'] = c

imgs = [gray, eq_global] + list(clahe_results.values())
titulos = ['Original', 'Eq. global'] + list(clahe_results.keys())

plt.figure(figsize=(14, 8))
for i, (im, t) in enumerate(zip(imgs, titulos)):
    plt.subplot(2, 3, i + 1)
    plt.imshow(im, cmap='gray', vmin=0, vmax=255)
    plt.title(t, fontsize=9); plt.axis('off')
plt.tight_layout(); plt.show()

# Comparacion de histogramas en dos plots para que se distingan bien
plt.figure(figsize=(13, 4))

plt.subplot(1, 2, 1)
plt.plot(histograma(gray), color='black', lw=2.5, label='Original')
plt.plot(histograma(eq_global), color='red', lw=2, ls='--', label='Eq. global')
plt.title('Original vs Ecualizacion global')
plt.xlabel('Intensidad'); plt.ylabel('N. pixeles')
plt.xlim(0, 255); plt.legend(); plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(histograma(gray), color='black', lw=2.5, label='Original')
plt.plot(histograma(clahe_results['clip=2.0, tile=(8, 8)']),
         color='tab:blue', lw=2, ls='-', label='CLAHE c=2.0 t=(8,8)')
plt.plot(histograma(clahe_results['clip=2.0, tile=(16, 16)']),
         color='tab:blue', lw=2, ls='--', label='CLAHE c=2.0 t=(16,16)')
plt.plot(histograma(clahe_results['clip=4.0, tile=(8, 8)']),
         color='tab:orange', lw=2, ls='-', label='CLAHE c=4.0 t=(8,8)')
plt.plot(histograma(clahe_results['clip=4.0, tile=(16, 16)']),
         color='tab:orange', lw=2, ls='--', label='CLAHE c=4.0 t=(16,16)')
plt.title('Original vs 4 configuraciones CLAHE')
plt.xlabel('Intensidad'); plt.ylabel('N. pixeles')
plt.xlim(0, 255); plt.legend(fontsize=8, loc='upper right'); plt.grid(alpha=0.3)

plt.tight_layout(); plt.show()

# Lectura: la eq. global extiende el histograma a todo [0,255] pero granula el fondo.
# CLAHE con clipLimit bajo (2.0, azul) realza sin ese ruido.
# CLAHE con clipLimit alto (4.0, naranja) logra mas contraste pero el grano vuelve a notarse.

In [ ]:
# --- Cuadricula de CLAHE sobre la imagen ---
# Cada celda se ecualiza con SU PROPIA CDF local, no con la de toda la imagen.

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for ax, tile in zip(axes, [(8, 8), (16, 16)]):
    overlay = img.copy()
    h, w = img.shape[:2]
    cell_h, cell_w = h // tile[0], w // tile[1]

    for i in range(1, tile[0]):
        y = i * cell_h
        cv2.line(overlay, (0, y), (w, y), (0, 0, 255), 2)
    for j in range(1, tile[1]):
        x = j * cell_w
        cv2.line(overlay, (x, 0), (x, h), (0, 0, 255), 2)

    ax.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
    ax.set_title(f'tileGridSize = {tile} -> {tile[0] * tile[1]} celdas')
    ax.axis('off')

plt.suptitle('Cada celda se ecualiza con su propia CDF local', y=1.02)
plt.tight_layout(); plt.show()

---
## Que pasa dentro de cada celda

Veamos 4 recortes de zonas distintas y como CLAHE los trata de forma independiente.


In [ ]:
h, w = img.shape[:2]
ps = 128

patches_pos = [
    ('Sup. izq.',  (slice(0, ps),        slice(0, ps))),
    ('Sup. der.',  (slice(0, ps),        slice(w - ps, w))),
    ('Inf. izq.',  (slice(h - ps, h),    slice(0, ps))),
    ('Inf. der.',  (slice(h - ps, h),    slice(w - ps, w))),
]

patches_gray = {nombre: cv2.cvtColor(img[y, x], cv2.COLOR_BGR2GRAY)
                 for nombre, (y, x) in patches_pos}

clahe_local = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
patches_eq = {nombre: clahe_local.apply(g) for nombre, g in patches_gray.items()}

nombres = list(patches_gray.keys())
colores = ['steelblue', 'seagreen', 'darkorange', 'mediumpurple']

fig, axes = plt.subplots(2, 4, figsize=(16, 6))

for i, nombre in enumerate(nombres):
    axes[0, i].imshow(cv2.cvtColor(img[patches_pos[i][1][0], patches_pos[i][1][1]],
                                     cv2.COLOR_BGR2RGB))
    axes[0, i].set_title(f'{nombre}\nantes')
    axes[0, i].axis('off')

    axes[1, i].imshow(patches_eq[nombre], cmap='gray', vmin=0, vmax=255)
    axes[1, i].set_title('despues CLAHE')
    axes[1, i].axis('off')

plt.suptitle('4 parches antes y despues de aplicar CLAHE localmente', y=1.02, fontsize=12)
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for color, nombre in zip(colores, nombres):
    axes[0].plot(histograma(patches_gray[nombre]), color=color,
                 label=nombre, alpha=0.85, lw=1.5)
    axes[1].plot(histograma(patches_eq[nombre]), color=color,
                 label=nombre, alpha=0.85, lw=1.5)

axes[0].set_title('Histogramas ANTES de CLAHE (4 parches superpuestos)')
axes[1].set_title('Histogramas DESPUES de CLAHE (4 parches superpuestos)')
for ax in axes:
    ax.set_xlabel('Intensidad'); ax.set_ylabel('N. pixeles')
    ax.set_xlim(0, 255); ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.tight_layout(); plt.show()

---
## Pregunta 1.1 - Sobre-amplificacion de ruido

> Por que la ecualizacion global arruina zonas homogeneas (pared lisa, fondo) y CLAHE no?

**Respuesta.** La ecualizacion global aplica una sola CDF a toda la imagen. En una zona homogenea (una pared lisa, un cielo plano, una sombra continua) la mayoria de pixeles tienen intensidades muy parecidas, asi que el histograma presenta ahi un **pico muy alto y estrecho**. Como la CDF es la integral acumulada del histograma, ese pico produce un **salto casi vertical** en la CDF, y la pendiente de la transformacion:

$$
\frac{dT}{dr} \propto p(r)
$$

se vuelve **enorme** justo en esa zona. El resultado:

- Cualquier pequena diferencia de entrada (ruido del sensor de $\pm 2$ niveles, invisible en la imagen original) se multiplica por esa pendiente enorme y se proyecta a **decenas de niveles** de salida.
- Lo que antes era una pared lisa se convierte en un campo de manchas de alto contraste: el ruido no se "inventa", se **amplifica**.
- Ademas, como la CDF es global, ese pico dominante **consume** gran parte del rango de salida $[0, 255]$, robandole cuantizacion al resto de la imagen.

**CLAHE** corrige esto en dos frentes complementarios:

1. **Recorte (clipping):** antes de calcular la CDF local, cualquier bin del histograma que supere `clipLimit` se **recorta** y el excedente se redistribuye uniformemente entre los demas bins. Esto acota la pendiente maxima de la CDF local y, por tanto, la maxima ganancia que se le puede dar al ruido.
2. **Localidad:** la CDF se calcula **por celdas** (`tileGridSize`), no sobre toda la imagen. Asi, el pico de una region homogenea solo gobierna la transformacion de esa region y no afecta al resto.

Esto es exactamente lo que se observa en el Ejercicio A.2: la ecualizacion global convierte las superficies lisas (la niebla, la piedra) en grano visible; CLAHE con `clipLimit = 2.0` mantiene esos fondos limpios y al subir a `4.0` el grano vuelve a notarse porque el tope de ganancia crece.


---
## Pregunta 1.2 - Canal de brillo vs RGB

> Por que es mala practica ecualizar R, G y B por separado y es mejor aplicar CLAHE solo sobre V (HSV) o L (CIELAB)?

**Respuesta.** El color que percibe el ojo no depende de los valores absolutos de R, G y B, sino de las **proporciones** entre ellos: las razones $R:G:B$ determinan el tono (hue) y la saturacion.

Si se aplica una ecualizacion **distinta** a cada canal (cada uno con su propia CDF), las razones $R:G:B$ cambian de manera arbitraria e independiente, y los tonos originales se destruyen: aparecen colores falsos que no existian en la escena (un cielo gris-azulado puede volverse magenta, una piel puede virar al azul, etc.). La ecualizacion independiente **confunde "redistribuir brillo" con "redistribuir color"**.

**Ejemplo concreto:** un pixel con $(R, G, B) = (100, 50, 50)$ representa un rojo oscuro. Si ecualizamos cada canal por separado podria terminar como $(200, 100, 60)$: las razones cambiaron y el pixel ya no es el mismo rojo.

Los espacios **HSV** y **CIELAB** resuelven esto al **separar** la informacion en dos partes:

- Un canal de **luminancia** (V en HSV, L en CIELAB) que concentra el brillo y la iluminacion de la escena.
- Canales de **crominancia** (H y S en HSV; a y b en CIELAB) que concentran la informacion de color.

La percepcion humana del contraste reside casi por completo en la luminancia. Aplicar CLAHE **solo** sobre V o L redistribuye el brillo (mejorando el contraste donde hace falta) **sin tocar las proporciones cromaticas**, asi que los tonos se preservan. CIELAB ademas es **perceptualmente uniforme**: variaciones iguales de L corresponden a diferencias de brillo percibido aproximadamente iguales, asi el realce queda mas acorde a la sensibilidad del ojo.

La celda siguiente lo demuestra empiricamente sobre `im1.png`: notese el viraje de color al ecualizar B, G, R por separado, frente a la preservacion del tono cuando CLAHE se aplica solo a V o solo a L.


In [ ]:
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

# (a) Mala practica: ecualizar B, G y R por separado
b, g, r = cv2.split(img)
eq_rgb = cv2.merge([cv2.equalizeHist(b), cv2.equalizeHist(g), cv2.equalizeHist(r)])

# (b) Buena practica: CLAHE solo sobre V (HSV)
h, s, v = cv2.split(hsv)
eq_hsv = cv2.cvtColor(cv2.merge([h, s, clahe.apply(v)]), cv2.COLOR_HSV2BGR)

# (c) Buena practica: CLAHE solo sobre L (CIELAB)
lab = cv2.cvtColor(img, cv2.COLOR_BGR2Lab)
L, A, B = cv2.split(lab)
eq_lab = cv2.cvtColor(cv2.merge([clahe.apply(L), A, B]), cv2.COLOR_Lab2BGR)

casos = [img, eq_rgb, eq_hsv, eq_lab]
titulos = ['Original', 'Eq. R,G,B independiente (tonos falsos)', 'CLAHE solo en V (HSV)', 'CLAHE solo en L (CIELAB)']

plt.figure(figsize=(12, 9))
for i, (im, t) in enumerate(zip(casos, titulos)):
    plt.subplot(2, 2, i + 1)
    plt.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
    plt.title(t, fontsize=10); plt.axis('off')
plt.tight_layout(); plt.show()

---
## Conclusiones del Modulo A

1. **Histograma como diagnostico.** El histograma es la PDF discreta de las intensidades de la imagen. Leerlo permite saber de un vistazo si la imagen es oscura, clara, o de contraste bajo antes de aplicar cualquier transformacion. En `im1.png` la masa en valores bajos indica sombras marcadas: caso ideal para ecualizar.

2. **Ecualizacion global: contraste maximo, ruido amplificado.** Usa una sola CDF para toda la imagen. Maximiza el contraste global pero, como la pendiente $ds/dr \propto p(r)$, cualquier pico en la PDF (zona homogenea) genera una pendiente enorme que amplifica el ruido de pocas unidades a decenas de niveles de salida. Resultado: zonas lisas granuladas.

3. **CLAHE: contraste local con ruido acotado.** Divide la imagen en celdas y aplica una CDF **local** con **clipping** del histograma segun `clipLimit`. Asi:
   - `clipLimit` controla la ganancia maxima (contraste vs ruido).
   - `tileGridSize` controla la escala de adaptacion (local vs global).
   Cada celda se ecualiza con SU propia estadistica, asi una zona oscura se realza segun su CDF, no segun la CDF de la imagen entera.

4. **Color: siempre sobre la luminancia.** En imagenes a color la ecualizacion debe operarse sobre el canal de luminancia (V en HSV o L en CIELAB), nunca sobre R, G, B por separado. Ecualizar canales independientes rompe las proporciones cromaticas que definen el tono y produce colores falsos. CIELAB es la mejor opcion cuando se busca fidelidad perceptual.
